
# Bengali NLP Safety — V19.1 Fixed mDeBERTa `is_toxic` OOF

**Protected Kaggle champion: V14 = 0.586**

V18 MuRIL did not pass the stability gate. V19 switches to a different architecture:
**`microsoft/mdeberta-v3-base`**.

V19 trains only `is_toxic`, because that label contributes **50% of the competition score**.
This keeps the larger model focused on the highest-value target.

### Design

- 5 grouped OOF folds over train + validation.
- mDeBERTa-v3 base, max length 128.
- 3 fixed epochs; average epoch 2 and 3 predictions.
- batch size 4 + gradient accumulation 4 for T4 memory.
- gradient checkpointing.
- mild class balancing for binary macro-F1.
- compares against the exact V14 toxic probability family.
- searches conservative mDeBERTa blends, script-aware blends, and a few small 3-way blends with V17 OOF models.
- V14 remains the fallback unless a candidate passes the fold-stability gate.

## Run

1. Open this notebook in Google Colab.
2. Select **T4 GPU**.
3. Run cell 1 and upload **`v19_data.zip`**.
4. Run the remaining cells.
5. Send **`v19_results.zip`** to ChatGPT before any Kaggle submission.


### V19.1 bug fix

The first V19 used mDeBERTa-v3 with gradient checkpointing. In this Colab/PyTorch/Transformers combination,
that can trigger:

`RuntimeError: Trying to backward through the graph a second time`

V19.1 disables gradient checkpointing and uses **batch size 2 + gradient accumulation 8**.
The effective batch remains 16 examples while staying safer on a T4.


In [ ]:

# 1. Upload V19 data
from google.colab import files
import pathlib, zipfile

uploaded=files.upload()
assert "v19_data.zip" in uploaded, "Upload the supplied v19_data.zip."

DATA_DIR=pathlib.Path("/content/v19_data")
DATA_DIR.mkdir(exist_ok=True)
with zipfile.ZipFile("v19_data.zip") as z:
    z.extractall(DATA_DIR)

print("Extracted:")
for p in sorted(DATA_DIR.iterdir()):
    print(" ",p.name,f"{p.stat().st_size:,} bytes")


In [ ]:

# 2. Dependencies + model preflight
import sys, subprocess
subprocess.check_call([
    sys.executable,"-m","pip","install","-q",
    "transformers==4.57.6","sentencepiece==0.2.1",
    "scikit-learn","pandas","numpy","tqdm","safetensors"
])

from transformers import AutoConfig, AutoTokenizer
MODEL_NAME="microsoft/mdeberta-v3-base"
print("Testing:",MODEL_NAME)
_ = AutoConfig.from_pretrained(MODEL_NAME)
_ = AutoTokenizer.from_pretrained(MODEL_NAME)
print("✅ mDeBERTa preflight passed.")


In [ ]:

# 3. Imports, settings, data
import gc, re, math, random, json, zipfile, unicodedata
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from sklearn.metrics import f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

SEED=42
N_FOLDS=5
MODEL_NAME="microsoft/mdeberta-v3-base"
MAX_LENGTH=128
EPOCHS=3
SNAPSHOT_EPOCHS=[2,3]
TRAIN_BATCH_SIZE=2
ACCUM_STEPS=8
EVAL_BATCH_SIZE=16
LR_BACKBONE=1e-5
LR_HEAD=2e-5
WEIGHT_DECAY=.01
WARMUP_RATIO=.10
MIN_GAIN=.003
MIN_WINS=3
MAX_LOSSES=1
LABELS=["is_toxic","implicit_toxicity","sarcasm","contains_profanity","threatening","identity_attack","sexual_content"]

def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False

seed_everything(SEED)
assert torch.cuda.is_available(), "Enable T4 GPU."

train=pd.read_csv(DATA_DIR/"train.csv",dtype={"ID":str})
valid=pd.read_csv(DATA_DIR/"validation.csv",dtype={"ID":str})
test=pd.read_csv(DATA_DIR/"test.csv",dtype={"ID":str})
v2v=pd.read_csv(DATA_DIR/"v2_valid.csv",dtype={"ID":str})
v2t=pd.read_csv(DATA_DIR/"v2_test.csv",dtype={"ID":str})
vf=pd.read_csv(DATA_DIR/"validation_folds.csv",dtype={"ID":str})
champ=pd.read_csv(DATA_DIR/"submission_v14_champion.csv",dtype={"ID":str})
with np.load(DATA_DIR/"old_xlmr_predictions.npz") as z:
    old_xlv=z["valid"].copy(); old_xlt=z["test"].copy()
with np.load(DATA_DIR/"v17_banglabert_oof_test.npz") as z:
    v17_bo=z["oof"].copy(); v17_bt=z["test"].copy()
with np.load(DATA_DIR/"v17_xlmr_oof_test.npz") as z:
    v17_xo=z["oof"].copy(); v17_xt=z["test"].copy()

assert len(train)==3297 and len(valid)==500 and len(test)==1000
assert champ["ID"].tolist()==test["ID"].tolist()
assert v2v["ID"].tolist()==valid["ID"].tolist()
assert v2t["ID"].tolist()==test["ID"].tolist()

def clean(x): return re.sub(r"\s+"," ",unicodedata.normalize("NFC",str(x))).strip()
def group_text(x): return re.sub(r"\s+"," ",unicodedata.normalize("NFKC",str(x))).strip().lower()
def has_bn(x): return bool(re.search(r"[\u0980-\u09FF]",str(x)))

combined=pd.concat([train,valid],ignore_index=True)
combined["clean_text"]=combined["text"].map(clean)
test["clean_text"]=test["text"].map(clean)
combined["group_text"]=combined["text"].map(group_text)
valid_has_bn=valid["text"].map(has_bn).to_numpy(bool)
test_has_bn=test["text"].map(has_bn).to_numpy(bool)

print("Combined rows:",len(combined),"test:",len(test))
print("is_toxic positives:",int(combined["is_toxic"].sum()))
print("Validation no-Bengali-script:",int((~valid_has_bn).sum()))


In [ ]:

# 4. Guaranteed grouped 5-fold assignment
gs=combined.groupby("group_text",sort=False).agg(
    size=("ID","size"),
    **{f"sum_{l}":(l,"sum") for l in LABELS}
).reset_index()

gp=np.array([combined[l].sum() for l in LABELS],float)
total=float(len(combined)); rarity=1/np.sqrt(np.maximum(gp,1))
mat=gs[[f"sum_{l}" for l in LABELS]].to_numpy(float)
rng=np.random.default_rng(SEED)
priority=(mat*rarity).sum(1)+.005*gs["size"].to_numpy()
order=np.argsort(-(priority+rng.random(len(gs))*1e-6))

fp=np.zeros((N_FOLDS,len(LABELS))); fs=np.zeros(N_FOLDS); assign={}
for idx in order:
    row=gs.iloc[idx]; vec=np.array([row[f"sum_{l}"] for l in LABELS],float); size=float(row["size"])
    costs=[]
    for f in range(N_FOLDS):
        p=fp.copy(); s=fs.copy(); p[f]+=vec; s[f]+=size
        costs.append(np.mean((p/np.maximum(gp[None,:],1)-1/N_FOLDS)**2)+.35*np.mean((s/total-1/N_FOLDS)**2))
    best=int(np.argmin(costs)); assign[row["group_text"]]=best; fp[best]+=vec; fs[best]+=size

combined["fold"]=combined["group_text"].map(assign).astype(int)
counts=combined.groupby("fold").size().reindex(range(N_FOLDS),fill_value=0)
pos=combined.groupby("fold")["is_toxic"].sum().reindex(range(N_FOLDS),fill_value=0)
assert (counts>0).all(); assert combined.groupby("group_text")["fold"].nunique().max()==1
assert counts.min()>=.8*(len(combined)/N_FOLDS) and counts.max()<=1.2*(len(combined)/N_FOLDS)
assert (pos>0).all()
print("✅ folds:",counts.to_dict())
print("toxic positives:",pos.to_dict())


In [ ]:

# 5. Dataset helpers
class ToxicDS(Dataset):
    def __init__(self,df,tok,labels=True):
        self.x=df["clean_text"].tolist(); self.y=df["is_toxic"].to_numpy(np.float32) if labels else None; self.tok=tok
    def __len__(self): return len(self.x)
    def __getitem__(self,i):
        z=self.tok(self.x[i],truncation=True,max_length=MAX_LENGTH,padding=False)
        if self.y is not None: z["labels"]=self.y[i]
        return z

class Collator:
    def __init__(self,tok): self.tok=tok
    def __call__(self,items):
        y=None
        if "labels" in items[0]: y=torch.tensor([x.pop("labels") for x in items],dtype=torch.float32)
        b=self.tok.pad(items,padding=True,pad_to_multiple_of=8,return_tensors="pt")
        if y is not None: b["labels"]=y
        return b

@torch.no_grad()
def predict(model,loader):
    assert len(loader.dataset)>0
    model.eval(); parts=[]
    for b in loader:
        b.pop("labels",None); inp={k:v.to("cuda",non_blocking=True) for k,v in b.items()}
        with torch.autocast(device_type="cuda",dtype=torch.float16): logits=model(**inp).logits.squeeze(-1)
        parts.append(torch.sigmoid(logits.float()).cpu().numpy())
    out=np.concatenate(parts); assert np.isfinite(out).all(); return out


In [ ]:

# 6. Train mDeBERTa 5-fold OOF
RUN_DIR=Path("/content/v19_output"); RUN_DIR.mkdir(exist_ok=True)
oof=np.zeros(len(combined),np.float32); test_probs=[]; history=[]
fold_seeds=[42,3407,2026,17,1337]
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)
probe=tokenizer(combined["clean_text"].sample(min(1000,len(combined)),random_state=SEED).tolist(),truncation=False)
lens=np.array([len(x) for x in probe["input_ids"]])
print("mDeBERTa p50/p95/p99:",np.percentile(lens,[50,95,99])); print("fraction >128:",float((lens>MAX_LENGTH).mean()))
del probe,lens

for fold in range(N_FOLDS):
    tr=combined[combined.fold!=fold].copy(); va=combined[combined.fold==fold].copy()
    assert set(tr["group_text"]).isdisjoint(set(va["group_text"]))
    cache=RUN_DIR/f"mdeberta_fold_{fold}.npz"; hist_path=RUN_DIR/f"mdeberta_fold_{fold}_history.csv"
    if cache.exists():
        with np.load(cache,allow_pickle=False) as z: idx=z["va_index"]; vp=z["valid"]; tp=z["test"]
        if np.array_equal(idx,va.index.to_numpy()) and len(vp)==len(va) and len(tp)==len(test):
            print("✅ reuse fold",fold); oof[idx]=vp; test_probs.append(tp)
            if hist_path.exists(): history.extend(pd.read_csv(hist_path).to_dict("records"))
            continue
    print("\n"+"="*70); print("fold",fold,"train",len(tr),"valid",len(va)); seed_everything(fold_seeds[fold])
    col=Collator(tokenizer)
    tl=DataLoader(ToxicDS(tr,tokenizer,True),batch_size=TRAIN_BATCH_SIZE,shuffle=True,num_workers=0,pin_memory=True,collate_fn=col)
    vl=DataLoader(ToxicDS(va,tokenizer,True),batch_size=EVAL_BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=True,collate_fn=col)
    el=DataLoader(ToxicDS(test,tokenizer,False),batch_size=EVAL_BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=True,collate_fn=col)
    model=AutoModelForSequenceClassification.from_pretrained(MODEL_NAME,num_labels=1).to("cuda")
    if hasattr(model, "gradient_checkpointing_disable"):
        model.gradient_checkpointing_disable()
    pos=float(tr["is_toxic"].sum()); neg=float(len(tr)-pos); pos_weight=np.sqrt(neg/max(pos,1))
    loss_fn=torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight],dtype=torch.float32,device="cuda"))
    print("pos_weight:",round(pos_weight,4))
    head=[]; backbone=[]
    for n,p in model.named_parameters():
        if not p.requires_grad: continue
        (head if ("classifier" in n or "pooler" in n) else backbone).append(p)
    optimizer=torch.optim.AdamW([
        {"params":backbone,"lr":LR_BACKBONE,"weight_decay":WEIGHT_DECAY},
        {"params":head,"lr":LR_HEAD,"weight_decay":WEIGHT_DECAY},
    ])
    steps=math.ceil(len(tl)/ACCUM_STEPS)*EPOCHS
    scheduler=get_linear_schedule_with_warmup(optimizer,int(steps*WARMUP_RATIO),steps)
    scaler=torch.amp.GradScaler("cuda"); vs=[]; ts=[]; fh=[]
    for epoch in range(1,EPOCHS+1):
        model.train(); optimizer.zero_grad(set_to_none=True); run=0.; seen=0
        for step,b in enumerate(tqdm(tl,desc=f"mDeBERTa f{fold} e{epoch}/{EPOCHS}")):
            yb=b.pop("labels").to("cuda",non_blocking=True); inp={k:v.to("cuda",non_blocking=True) for k,v in b.items()}
            with torch.autocast(device_type="cuda",dtype=torch.float16): logits=model(**inp).logits.squeeze(-1)
            loss=loss_fn(logits.float(),yb)
            if not torch.isfinite(loss): raise FloatingPointError("non-finite loss")
            scaler.scale(loss/ACCUM_STEPS).backward(); run+=float(loss.item())*len(yb); seen+=len(yb)
            if (step+1)%ACCUM_STEPS==0 or step+1==len(tl):
                scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(),1.)
                old=scaler.get_scale(); scaler.step(optimizer); scaler.update()
                if scaler.get_scale()>=old: scheduler.step()
                optimizer.zero_grad(set_to_none=True)
        rec={"fold":fold,"epoch":epoch,"loss":run/max(seen,1)}; print(rec); fh.append(rec); pd.DataFrame(fh).to_csv(hist_path,index=False)
        if epoch in SNAPSHOT_EPOCHS: vs.append(predict(model,vl)); ts.append(predict(model,el))
    vp=np.mean(vs,axis=0); tp=np.mean(ts,axis=0); oof[va.index.to_numpy()]=vp; test_probs.append(tp); history.extend(fh)
    np.savez_compressed(cache,va_index=va.index.to_numpy(),valid=vp,test=tp)
    del model,tl,vl,el,optimizer,scheduler,scaler,backbone,head; gc.collect(); torch.cuda.empty_cache()

assert len(test_probs)==N_FOLDS
mdeberta_test=np.mean(test_probs,axis=0)
np.savez_compressed(RUN_DIR/"mdeberta_oof_test.npz",oof=oof,test=mdeberta_test)
pd.DataFrame(history).to_csv(RUN_DIR/"mdeberta_training_history.csv",index=False)
print("✅ mDeBERTa OOF complete.")


In [ ]:

# 7. Reconstruct V14 toxic baseline + OOF validation
def logit(p):
    p=np.clip(np.asarray(p,float),1e-6,1-1e-6); return np.log(p/(1-p))
def sigmoid(z): return 1/(1+np.exp(-z))
basev=.875*v2v["is_toxic_prob"].to_numpy(float)+.125*old_xlv[1:3,:,0].mean(0)
baset=.875*v2t["is_toxic_prob"].to_numpy(float)+.125*old_xlt[1:3,:,0].mean(0)
orig_idx=np.arange(len(train),len(train)+len(valid)); mdv=oof[orig_idx]
v17v=.5*v17_bo[orig_idx]+.5*v17_xo[orig_idx]; v17t=.5*v17_bt+.5*v17_xt
folds=valid["ID"].map(vf.set_index("ID")["fold"]).to_numpy(int); y=valid["is_toxic"].to_numpy(int)
assert set(folds)==set(range(5)); print("Prepared OOF validation.")


In [ ]:

# 8. Binary macro-F1 cross-fitted thresholds
def macro(y,pred): return float(f1_score(y,pred,average="macro",labels=[0,1],zero_division=0))
def best_t(y,p):
    vals=np.unique(p)
    if len(vals)>600: vals=np.quantile(vals,np.linspace(0,1,600))
    cand=np.unique(np.r_[0.,vals,1.000001]); s=np.array([macro(y,p>=t) for t in cand]); return float(np.median(cand[np.isclose(s,s.max(),atol=1e-12,rtol=0)]))
def cv(p):
    pred=np.zeros(len(y),np.int8); fs=[]; ts=[]
    for f in range(5):
        h=folds==f; t=best_t(y[~h],p[~h]); ts.append(t); pred[h]=(p[h]>=t).astype(np.int8); fs.append(macro(y[h],pred[h]))
    return {"score":macro(y,pred),"fold_scores":np.array(fs),"thresholds":np.array(ts),"prediction":pred}
basecv=cv(basev); mdcv=cv(mdv)
print("V14 toxic:",basecv["score"],np.round(basecv["fold_scores"],4))
print("Pure mDeBERTa:",mdcv["score"],np.round(mdcv["fold_scores"],4))


In [ ]:

# 9. Search conservative mDeBERTa ensembles
rows=[]; best=None; candidates=[]
for mode in ["prob","logit"]:
    for w_i in range(1,13):
        w=w_i/20; p=(1-w)*basev+w*mdv if mode=="prob" else sigmoid((1-w)*logit(basev)+w*logit(mdv))
        candidates.append({"kind":"base_mdeberta","mode":mode,"w_md":w,"prob":p})
for mode in ["prob","logit"]:
    for w_bn in [0,.1,.2,.3,.4]:
        for w_non in [.1,.2,.3,.4,.5,.6]:
            if abs(w_non-w_bn)>.4: continue
            ww=np.where(valid_has_bn,w_bn,w_non); p=(1-ww)*basev+ww*mdv if mode=="prob" else sigmoid((1-ww)*logit(basev)+ww*logit(mdv))
            candidates.append({"kind":"script_aware","mode":mode,"w_bn":w_bn,"w_non":w_non,"prob":p})
for mode in ["prob","logit"]:
    for wb,wm,wv in [(.8,.15,.05),(.75,.2,.05),(.75,.15,.10),(.7,.2,.10),(.7,.25,.05),(.65,.25,.10),(.6,.3,.10)]:
        p=wb*basev+wm*mdv+wv*v17v if mode=="prob" else sigmoid(wb*logit(basev)+wm*logit(mdv)+wv*logit(v17v))
        candidates.append({"kind":"plus_v17","mode":mode,"w_base":wb,"w_md":wm,"w_v17":wv,"prob":p})
for c in candidates:
    p=c.pop("prob"); res=cv(p); wins=int((res["fold_scores"]>basecv["fold_scores"]+1e-12).sum()); losses=int((res["fold_scores"]<basecv["fold_scores"]-1e-12).sum()); ties=5-wins-losses; gain=res["score"]-basecv["score"]
    passes=(gain>=MIN_GAIN and wins>=MIN_WINS and losses<=MAX_LOSSES)
    rec={**c,"score":res["score"],"baseline_score":basecv["score"],"gain":gain,"wins":wins,"ties":ties,"losses":losses,"passes_gate":passes,"threshold":float(np.median(res["thresholds"]))}; rows.append(rec)
    intervention=rec.get("w_md",(rec.get("w_bn",0)+rec.get("w_non",0))/2 if rec["kind"]=="script_aware" else 1-rec.get("w_base",1))
    key=(passes,res["score"],wins,-losses,-intervention)
    if best is None or key>best["key"]: best={"key":key,"rec":rec,"cv":res}
search=pd.DataFrame(rows).sort_values(["passes_gate","gain","wins","losses"],ascending=[False,False,False,True]); search.to_csv(RUN_DIR/"v19_ensemble_search.csv",index=False)
display(search.head(30).round(5)); print("BEST:",best["rec"])


In [ ]:

# 10. Script diagnostics
report=[]
for name,mask in [("Bengali script",valid_has_bn),("No Bengali script",~valid_has_bn)]:
    report.append({"subset":name,"rows":int(mask.sum()),"positives":int(y[mask].sum()),"V14_macro_f1":macro(y[mask],basecv["prediction"][mask]),"mDeBERTa_macro_f1":macro(y[mask],mdcv["prediction"][mask])})
script=pd.DataFrame(report); script.to_csv(RUN_DIR/"v19_script_report.csv",index=False); display(script.round(4))


In [ ]:

# 11. Controlled submission only if the gate passes
rec=best["rec"]; passes=bool(rec["passes_gate"])
if rec["kind"]=="base_mdeberta":
    w=float(rec["w_md"]); pt=(1-w)*baset+w*mdeberta_test if rec["mode"]=="prob" else sigmoid((1-w)*logit(baset)+w*logit(mdeberta_test))
elif rec["kind"]=="script_aware":
    ww=np.where(test_has_bn,float(rec["w_bn"]),float(rec["w_non"])); pt=(1-ww)*baset+ww*mdeberta_test if rec["mode"]=="prob" else sigmoid((1-ww)*logit(baset)+ww*logit(mdeberta_test))
else:
    wb=float(rec["w_base"]); wm=float(rec["w_md"]); wv=float(rec["w_v17"]); pt=wb*baset+wm*mdeberta_test+wv*v17t if rec["mode"]=="prob" else sigmoid(wb*logit(baset)+wm*logit(mdeberta_test)+wv*logit(v17t))
pred=(pt>=float(rec["threshold"])).astype(np.int8); sub=champ.copy(); before=sub["is_toxic"].to_numpy(int)
if passes: sub["is_toxic"]=pred
changed=int((before!=sub["is_toxic"].to_numpy(int)).sum()); sub.to_csv(RUN_DIR/"submission_v19_toxic.csv",index=False)
report={"version":"v19_mdeberta_toxic","champion_score":0.586,"pure_mdeberta_crossfit":mdcv["score"],"baseline_crossfit":basecv["score"],"selected":rec,"passes_gate":passes,"changed_rows":changed,"warning":"Send v19_results.zip to ChatGPT before Kaggle submission."}
(RUN_DIR/"v19_report.json").write_text(json.dumps(report,indent=2)); print(report)


In [ ]:

# 12. Package results
zip_path=Path("/content/v19_results.zip")
with zipfile.ZipFile(zip_path,"w",compression=zipfile.ZIP_DEFLATED) as z:
    for p in RUN_DIR.iterdir():
        if p.is_file(): z.write(p,arcname=p.name)
print("Created:",zip_path)
from google.colab import files
files.download(str(zip_path))
